# 15SW Safety — Weekly Base Rate

Updates the **Weekly base rate** panel on the app's dashboard: how often a normal
week has a flight mishap, plus the conditions to brief each week.

### Every week
Press **▶** on **Update the 15SW Safety app** below. It reads the mishap records
from the app, works out the base rate and each week's conditions, and saves them
back to the app. It takes about 2 minutes and prints a summary when it's done.

### First time only (Google Colab)
1. Go to [colab.research.google.com](https://colab.research.google.com) → *File → Upload notebook* → pick this file.
2. Click the **key icon** (Secrets) in the left bar and add two secrets, each with *Notebook access* on:
   - `APP_URL` — the live site, e.g. `https://safety.yourdomain.com`
   - `MODEL_API_TOKEN` — the same value as `MODEL_API_TOKEN` in the app's `.env`

The app only sends what this needs (mishap dates, type, flight/ground, category,
aircraft, sortie dates) — never descriptions or names. On a laptop without those
two settings, it reads and writes the local database instead.

The **SPI** (rolling 90-day count on the dashboard) is calculated live by the app
itself; this notebook doesn't touch it.


In [ ]:
# @title ▶ Update the 15SW Safety app
# @markdown Set the options, then press ▶ on the left. About 2 minutes; a summary prints at the end.

# @markdown **Which weeks count as "normal"** for the base rate:
BASE_RATE_PERIOD = "Last 5 years"  # @param ["Last 5 years", "Last 3 years", "All history"]
# @markdown **El Niño right now** — used for recent months NOAA hasn't published yet (see the latest PAGASA bulletin):
ENSO_NOW = "El Niño"  # @param ["El Niño", "La Niña", "Neutral", "Latest NOAA value"]
# @markdown **Weeks ahead to fill in**, so the dashboard still shows "this week" if an update is missed:
WEEKS_AHEAD = 8  # @param {type:"slider", min:0, max:12, step:1}

import os, sqlite3, json
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ── Where the data lives ───────────────────────────────────────────────────
# ONLINE (Colab or any PC): records come from the live app and results go back
# through its token-protected link (/api/model/*). Needs APP_URL and
# MODEL_API_TOKEN, from Colab Secrets or environment variables.
# OFFLINE (laptop, no settings): the local SQLite database is used instead.
def _setting(name):
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata   # only exists inside Colab
            value = userdata.get(name) or ""
        except Exception:
            pass
    return value.strip()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

APP_URL = _setting("APP_URL").rstrip("/")
MODEL_API_TOKEN = _setting("MODEL_API_TOKEN")
ONLINE = bool(APP_URL and MODEL_API_TOKEN)
if IN_COLAB and not ONLINE:
    raise RuntimeError("Add APP_URL and MODEL_API_TOKEN under Secrets (key icon, left bar), "
                       "turn on Notebook access for both, then press ▶ again.")

def api(method, path, **kwargs):
    """Call the app's notebook link; stops with the app's message on failure."""
    r = requests.request(
        method, f"{APP_URL}/api/model/{path}", timeout=120,
        headers={"Authorization": f"Bearer {MODEL_API_TOKEN}",
                 "X-Model-Token": MODEL_API_TOKEN,   # some hosts strip Authorization
                 "Accept": "application/json"},
        **kwargs)
    if not r.ok:
        raise RuntimeError(f"App link {method} {path} failed ({r.status_code}): {r.text[:300]}")
    return r.json()

def find_db():
    for p in [Path("../database/database.sqlite"), Path("database/database.sqlite")]:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError("No APP_URL / MODEL_API_TOKEN set and no local database/database.sqlite found.")

# Airports near the bases that publish METAR. The base fields themselves (and
# Laguindingan RPMY) return nothing from this archive.
STATIONS = {"RPLL": "Manila (Sangley / MDAAB)", "RPMZ": "Zamboanga (EAAB)", "RPMD": "Davao"}

ENSO_SETTING = {"El Niño": 1.0, "La Niña": -1.0, "Neutral": 0.0, "Latest NOAA value": None}[ENSO_NOW]
BASE_YEARS = {"Last 5 years": 5, "Last 3 years": 3, "All history": None}[BASE_RATE_PERIOD]

# Weeks run Monday–Sunday, in Philippine time (Colab's clock is UTC).
TODAY = pd.Timestamp.now(tz="Asia/Manila").tz_localize(None).normalize()
THIS_WEEK = TODAY.to_period("W-SUN").start_time

def monday(values):
    return pd.to_datetime(values).dt.to_period("W-SUN").dt.start_time

# ── Inputs ─────────────────────────────────────────────────────────────────
def load_inputs():
    if ONLINE:
        data = api("GET", "data")
        rec = pd.DataFrame(data["mishaps"])
        sor = pd.DataFrame(data["sorties"], columns=["flight_date"])
    else:
        con = sqlite3.connect(find_db())
        try:
            rec = pd.read_sql_query("SELECT mishap_date, environment FROM mishaps", con)
            try:
                sor = pd.read_sql_query("SELECT flight_date FROM flight_schedules", con)
            except Exception:
                sor = pd.DataFrame(columns=["flight_date"])
        finally:
            con.close()
    rec["mishap_date"] = pd.to_datetime(rec["mishap_date"])
    sor["flight_date"] = pd.to_datetime(sor["flight_date"])
    return rec, sor

def fetch_weather(start):
    """Weekly weather across the stations: worst visibility, haze and thunderstorm hours, max wind."""
    frames = []
    for st in STATIONS:
        try:
            r = requests.get("https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py", params={
                "station": st, "data": ["vsby", "sknt", "wxcodes"],
                "year1": start.year, "month1": start.month, "day1": start.day,
                "year2": TODAY.year, "month2": TODAY.month, "day2": TODAY.day,
                "tz": "Etc/UTC", "format": "onlycomma", "missing": "M", "trace": "T", "latlon": "no",
            }, timeout=180)
            r.raise_for_status()
            frames.append(pd.read_csv(StringIO(r.text), na_values=["M", "T"]))
        except Exception as e:
            print(f"     weather {st}: skipped ({e.__class__.__name__})")
    if not frames:
        return pd.DataFrame({"week": pd.Series(dtype="datetime64[ns]"), "vsby_min": np.nan,
                             "haze_hours": np.nan, "ts_hours": np.nan, "wind_max": np.nan})
    m = pd.concat(frames, ignore_index=True)
    m["week"] = monday(m["valid"])
    codes = m["wxcodes"].fillna("")
    m["haze"] = codes.str.contains("HZ")
    m["ts"] = codes.str.contains("TS")          # thunderstorm
    for c in ("vsby", "sknt"):
        m[c] = pd.to_numeric(m[c], errors="coerce")
    return m.groupby("week").agg(vsby_min=("vsby", "min"), haze_hours=("haze", "sum"),
                                 ts_hours=("ts", "sum"), wind_max=("sknt", "max")).reset_index()

def fetch_oni():
    """NOAA Oceanic Niño Index: {(year, month): value}. ≥ +0.5 El Niño, ≤ −0.5 La Niña."""
    center = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
              "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
    out = {}
    try:
        text = requests.get("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt", timeout=60).text
        for line in text.splitlines()[1:]:
            p = line.split()
            if len(p) >= 4 and p[0] in center:
                out[(int(p[1]), center[p[0]])] = float(p[3])
    except Exception as e:
        print(f"     El Niño index: skipped ({e.__class__.__name__}), using your setting")
    return out

def oni_value(oni, year, month):
    v = oni.get((year, month))
    if v is not None:
        return v
    if ENSO_SETTING is not None:            # months NOAA hasn't published yet
        return ENSO_SETTING
    return oni[max(oni)] if oni else 0.0

def monsoon_phase(month):
    if month in (6, 7, 8, 9):   return "southwest"   # Habagat - afternoon storms
    if month in (11, 12, 1, 2): return "northeast"   # Amihan
    if month in (3, 4, 5):      return "pre"         # hot/dry build-up
    return "post"                                     # October transition

def bird_windows(month):
    # East Asian-Australasian Flyway: southbound peak Sep–Nov, northbound Feb–May.
    return int(month in (9, 10, 11)), int(month in (2, 3, 4, 5))

# ── One row per week ───────────────────────────────────────────────────────
def build_weeks(rec, sor, weather, oni):
    last = THIS_WEEK + pd.Timedelta(weeks=WEEKS_AHEAD)
    weeks = pd.DataFrame({"week": pd.date_range(monday(rec["mishap_date"]).min(), last, freq="W-MON")})
    flight_counts = monday(rec.loc[rec["environment"] == "flight", "mishap_date"]).value_counts()
    weeks["mishaps"] = weeks["week"].map(flight_counts).fillna(0).astype(int)
    weeks["recent_mishaps"] = weeks["mishaps"].shift(1).rolling(4, min_periods=1).sum().fillna(0)
    weeks.loc[weeks["week"] > THIS_WEEK, "recent_mishaps"] = np.nan   # not known yet
    month = weeks["week"].dt.month
    weeks["monsoon_phase"] = month.map(monsoon_phase)
    for name in ("southwest", "northeast", "pre"):
        weeks[f"{name}_monsoon"] = (weeks["monsoon_phase"] == name).astype(int)
    birds = month.map(bird_windows)
    weeks["bird_south"] = [b[0] for b in birds]
    weeks["bird_north"] = [b[1] for b in birds]
    weeks["oni"] = [oni_value(oni, w.year, w.month) for w in weeks["week"]]
    weeks["el_nino"] = (weeks["oni"] >= 0.5).astype(int)
    weeks = weeks.merge(weather, on="week", how="left")
    if sor.empty:
        weeks["sorties"] = np.nan
        return weeks
    sorties = monday(sor["flight_date"]).value_counts().rename("sorties").rename_axis("week").reset_index()
    return weeks.merge(sorties, on="week", how="left")

# ── The base rate ──────────────────────────────────────────────────────────
def base_rate(weeks):
    """Share of completed weeks in the period with at least one flight mishap."""
    past = weeks[weeks["week"] < THIS_WEEK]
    if BASE_YEARS:
        past = past[past["week"] >= THIS_WEEK - pd.DateOffset(years=BASE_YEARS)]
    hits = int((past["mishaps"] > 0).sum())
    return hits / len(past), hits, len(past), past

# ── Conditions to brief ────────────────────────────────────────────────────
def hist_week(rec, week):
    """Mishaps (flight + ground) that fell in this same calendar week in any year."""
    doy = rec["mishap_date"].dt.dayofyear
    sub = rec[((doy - week.dayofyear) % 365) <= 6]
    return len(sub), int(sub["mishap_date"].dt.year.nunique())

def conditions(row, rec, norms):
    items = []   # (text, tone, weight)
    hc, hy = hist_week(rec, row["week"])
    if hc > 0:
        items.append((f"{hc} past mishaps in this calendar week (over {hy} yrs)", "info", min(8 * hc, 30)))
    if row["bird_south"]:
        items.append(("Bird migration - southbound peak", "alert", 18))
    elif row["bird_north"]:
        items.append(("Bird migration - northbound passage", "alert", 14))
    oni = float(row["oni"])
    if oni >= 0.5:
        strength = "strong" if oni >= 1.5 else "moderate"
        items.append((f"El Nino ({strength}, ONI {oni:+.1f}) - drier, hazier, more shear", "alert",
                      12 + int(min(oni, 2.5) * 4)))
    elif oni <= -0.5:
        items.append((f"La Nina (ONI {oni:+.1f}) - wetter, more storms", "info", 12))
    phase = row["monsoon_phase"]
    if phase == "southwest":
        items.append(("Southwest monsoon (Habagat) - afternoon storms", "info", 12))
    elif phase == "northeast":
        items.append(("Northeast monsoon (Amihan)", "info", 8))
    else:
        items.append(("Monsoon transition - unsettled, less predictable", "alert", 14))
    ts = row.get("ts_hours")
    if pd.notna(ts) and ts > norms["ts_hours"] + 2:
        items.append(("Thunderstorms reported - convective turbulence", "alert", 22))
    srt = row.get("sorties")
    if pd.notna(srt) and srt > 0:
        items.append((f"{int(srt)} sorties scheduled this week", "info", min(10 + int(srt), 26)))
    recent = row.get("recent_mishaps")
    if pd.notna(recent) and recent >= 1:
        items.append((f"{int(recent)} recent mishap(s) in the last 4 weeks", "info", min(8 * int(recent), 24)))
    vm = row.get("vsby_min")
    if pd.notna(vm) and 0 < vm < norms["vsby_min"] * 0.6:
        items.append(("Low visibility / haze", "alert", 20))
    wm = row.get("wind_max")
    if pd.notna(wm) and wm > norms["wind_max"] + 8:
        items.append(("Strong winds", "info", 10))
    return items

def score_week(row, rate, rec, norms, phrase):
    items = sorted(conditions(row, rec, norms), key=lambda t: -t[2])
    if items:
        top = max(w for _, _, w in items)
        reasons = [{"text": t, "tone": tone, "impact": int(round(100 * w / top))} for t, tone, w in items[:5]]
    else:
        reasons = [{"text": "Conditions look normal for this week.", "tone": "good", "impact": 0}]
    pct = int(round(rate * 100))
    return {
        "week_start": row["week"].strftime("%Y-%m-%d"),
        "risk_level": "baseline",
        "likelihood": pct,
        "baseline": pct,
        "headline": f"{phrase}, about {pct}% of weeks had a flight mishap. {len(items)} condition(s) to brief this week.",
        "reasons": reasons,
    }

# ── Save ───────────────────────────────────────────────────────────────────
def save(rows, source):
    rows = json.loads(json.dumps(rows, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    if ONLINE:
        return api("POST", "forecasts", json={"source": source, "forecasts": rows})["saved"]
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    con = sqlite3.connect(find_db())
    try:
        for r in rows:
            # One wing-wide row per week (base NULL): replace, don't pile up.
            con.execute("DELETE FROM safety_forecasts WHERE date(week_start)=date(?) AND base IS NULL",
                        (r["week_start"],))
            con.execute("INSERT INTO safety_forecasts (week_start, base, risk_level, likelihood, baseline, "
                        "headline, reasons, source, generated_at, created_at, updated_at) "
                        "VALUES (?, NULL, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                        (r["week_start"], r["risk_level"], r["likelihood"], r["baseline"], r["headline"],
                         json.dumps(r["reasons"]), source, now, now, now))
        con.commit()
    finally:
        con.close()
    return len(rows)

# ── Run ────────────────────────────────────────────────────────────────────
def run():
    global RECORDS, WEEKS
    print("15SW Safety - weekly base rate update")
    print(f"  Using: {APP_URL if ONLINE else 'the local database (offline)'}")
    rec, sor = load_inputs()
    print(f"  Records: {len(rec)} mishaps ({int((rec['environment'] == 'flight').sum())} flight), "
          f"{len(sor)} scheduled sorties")

    first = monday(rec["mishap_date"]).min()
    norm_start = first if BASE_YEARS is None else max(first, THIS_WEEK - pd.DateOffset(years=BASE_YEARS))
    print("  Downloading airport weather (Iowa State METAR archive)...")
    weather = fetch_weather(min(norm_start, pd.Timestamp(TODAY.year, 1, 1)))
    print("  Loading the El Nino index (NOAA)...")
    oni = fetch_oni()

    weeks = build_weeks(rec, sor, weather, oni)
    rate, hits, n, past = base_rate(weeks)
    norms = past[["ts_hours", "vsby_min", "wind_max"]].median()
    phrase = f"Over the {BASE_RATE_PERIOD.lower()}" if BASE_YEARS else f"Since {first.year}"

    chosen = weeks[(weeks["week"].dt.year == TODAY.year) | (weeks["week"] > THIS_WEEK)]
    rows = [score_week(r, rate, rec, norms, phrase) for _, r in chosen.iterrows()]
    saved = save(rows, source=f"Notebook · base rate {BASE_RATE_PERIOD.lower()}")

    this_week = next(r for r in rows if r["week_start"] == THIS_WEEK.strftime("%Y-%m-%d"))
    print()
    print(f"BASE RATE ({BASE_RATE_PERIOD.lower()}): {rate:.1%}")
    print(f"  {hits} of {n} weeks since {past['week'].min():%d %b %Y} had at least one flight mishap.")
    print(f"THIS WEEK ({THIS_WEEK:%d %b} - {THIS_WEEK + pd.Timedelta(days=6):%d %b %Y}) - conditions to brief:")
    for r in this_week["reasons"]:
        print(f"  - {r['text']}")
    print()
    print(f"Saved {saved} weeks ({chosen['week'].min():%d %b %Y} to {chosen['week'].max():%d %b %Y}) "
          f"to {APP_URL or 'the local database'}.")
    if ONLINE:
        print(f"Open the dashboard: {APP_URL}/")
    RECORDS, WEEKS = rec, weeks

run()


## How the numbers are made

**Base rate.** The calendar is split into Monday–Sunday weeks. A week counts if it
had at least one flight mishap. Base rate = counted weeks ÷ all completed weeks in
the period you picked above. It's the same for every week: the background rate,
not a prediction.

**Conditions to brief.** Rules, each shown only when it applies:
- past mishaps (flight or ground) in the same calendar week in earlier years
- bird migration — southbound Sep–Nov, northbound Feb–May
- El Niño / La Niña — NOAA's index, or your setting for months NOAA hasn't published
- monsoon phase — Habagat, Amihan or the transition months
- thunderstorms, low visibility or strong winds compared with normal (airport weather)
- sorties scheduled (from the Flight Order PDFs) and mishaps in the last 4 weeks

**Weeks ahead** only get what's known in advance (calendar, season, El Niño). They're
replaced with full data the next time you press ▶.

**Why no prediction?** Testing found that nothing — season, El Niño, weather, birds,
recent mishaps — predicts a given week better than the base rate. The *Model check*
below re-runs that test.


## Look closer (optional)

In [ ]:
# @title Base rate by year
# @markdown Share of weeks with at least one flight mishap, per year. Run the update above first.
past = WEEKS[WEEKS["week"] < THIS_WEEK]
by_year = (past.assign(year=past["week"].dt.year, hit=past["mishaps"] > 0)
               .groupby("year").agg(weeks=("hit", "size"), weeks_with_a_flight_mishap=("hit", "sum")))
by_year["share"] = (by_year["weeks_with_a_flight_mishap"] / by_year["weeks"]).map("{:.0%}".format)
by_year


In [ ]:
# @title Model check — does anything beat the base rate?
# @markdown Walk-forward test: train only on past weeks, predict the next one, repeat. Weather is only
# @markdown downloaded for the base-rate period, so pick "All history" above for the full test.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["oni", "el_nino", "southwest_monsoon", "northeast_monsoon", "pre_monsoon",
            "bird_south", "bird_north", "recent_mishaps",
            "vsby_min", "haze_hours", "ts_hours", "wind_max", "sorties"]
d = WEEKS[WEEKS["week"] < THIS_WEEK].copy()
usable = [f for f in FEATURES if d[f].notna().mean() >= 0.6 and d[f].nunique(dropna=True) > 1]
print("Testing:", ", ".join(usable))
X = d[usable].fillna(d[usable].median()).fillna(0).to_numpy(float)
y = (d["mishaps"] > 0).astype(int).to_numpy()

oof = np.full(len(y), np.nan)
for i in range(150, len(y)):
    if len(np.unique(y[:i])) == 2:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000))
        oof[i] = model.fit(X[:i], y[:i]).predict_proba(X[i:i + 1])[0, 1]
mask = ~np.isnan(oof)
rate = y[mask].mean()
auc = roc_auc_score(y[mask], oof[mask])
skill = 1 - brier_score_loss(y[mask], oof[mask]) / brier_score_loss(y[mask], np.full(mask.sum(), rate))
print(f"Weeks tested: {int(mask.sum())}   AUC {auc:.3f} (0.50 = chance)   "
      f"Brier skill {skill:+.1%} (above 0 = beats the base rate)")
print("Verdict:", "a model beats the base rate - worth a closer look." if skill > 0
      else "nothing beats the base rate, so the app keeps showing it.")
